# 05 – Place Profile Generation
## Aspect Detection + Sentiment Classification valós Google Local review-kon

**Bemeneti fájlok:**
- `aspect_distilbert_weighted.zip`
- `sentiment_distilbert_application.zip`
- `selected_places.csv`
- `selected_reviews.csv`
- `selected_review_sentences.csv`

**Pipeline:**

Google review-mondat  
→ Aspect Detection  
→ észlelt aspektus(ok)  
→ Aspect-Level Sentiment Classification  
→ review-szintű aggregálás  
→ helyszintű place profile

A hosszabb review-k torzító hatásának csökkentése érdekében először review-szinten aggregálunk, és csak utána helyszinten.


## 1. Környezet

In [ ]:
!pip -q install "transformers>=4.46,<5" pandas numpy

In [ ]:
from pathlib import Path
import shutil
import gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Az öt input fájl feltöltése

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Nem Google Colab környezet.")


In [ ]:
ASPECT_ZIP = Path("aspect_distilbert_weighted.zip")
SENTIMENT_ZIP = Path("sentiment_distilbert_application.zip")
PLACES_PATH = Path("selected_places.csv")
REVIEWS_PATH = Path("selected_reviews.csv")
SENTENCES_PATH = Path("selected_review_sentences.csv")

for path in [ASPECT_ZIP, SENTIMENT_ZIP, PLACES_PATH, REVIEWS_PATH, SENTENCES_PATH]:
    assert path.exists(), f"Hiányzó fájl: {path}"

print("Mind az öt input fájl elérhető.")


## 3. Modellek kicsomagolása

In [ ]:
ASPECT_MODEL_DIR = Path("aspect_model")
SENTIMENT_MODEL_DIR = Path("sentiment_model")

for folder in [ASPECT_MODEL_DIR, SENTIMENT_MODEL_DIR]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir()

shutil.unpack_archive(ASPECT_ZIP, ASPECT_MODEL_DIR)
shutil.unpack_archive(SENTIMENT_ZIP, SENTIMENT_MODEL_DIR)

print("Aspect model:", sorted(p.name for p in ASPECT_MODEL_DIR.iterdir()))
print("Sentiment model:", sorted(p.name for p in SENTIMENT_MODEL_DIR.iterdir()))


## 4. Adatok betöltése

In [ ]:
places_df = pd.read_csv(PLACES_PATH)
reviews_df = pd.read_csv(REVIEWS_PATH)
sentences_df = pd.read_csv(SENTENCES_PATH)

print("Helyek:", len(places_df))
print("Review-k:", len(reviews_df))
print("Mondatok:", len(sentences_df))

assert len(places_df) == 20
assert sentences_df["sentence"].notna().all()
assert sentences_df["sentence_id"].is_unique

display(places_df.head())
display(sentences_df.head())


# 5. Aspect Detection

A végleges threshold **0.55**, amelyet a SemEval validation adaton választottunk ki. A Google Local adatokon ezt nem hangoljuk újra.


### Deployment threshold

A modellválasztás során a **validation threshold 0.55** volt. A Google Local alkalmazási adatokon végzett domain sanity check alapján azonban ez túl sok aspektust jelölt mondatonként, ezért az alkalmazási pipeline-ban konzervatívabb **deployment threshold = 0.65** értéket használunk.

Ez nem új supervised tuning a gold testre, hanem külön deployment/domain calibration az annotálatlan Google review-kon.

In [ ]:
ASPECT_THRESHOLD = 0.65
ASPECT_BATCH_SIZE = 64

aspect_tokenizer = AutoTokenizer.from_pretrained(
    ASPECT_MODEL_DIR,
    local_files_only=True,
)

aspect_model = AutoModelForSequenceClassification.from_pretrained(
    ASPECT_MODEL_DIR,
    local_files_only=True,
).to(DEVICE)

aspect_model.eval()

aspect_id2label = {
    int(k): v for k, v in aspect_model.config.id2label.items()
}
ASPECTS = [aspect_id2label[i] for i in range(len(aspect_id2label))]

print("Aspektusok:", ASPECTS)


In [ ]:
def batched_aspect_probabilities(texts, batch_size=64):
    output = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = aspect_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.inference_mode():
            logits = aspect_model(**inputs).logits
            probs = torch.sigmoid(logits).cpu().numpy()

        output.append(probs)

    return np.vstack(output)

sentence_texts = sentences_df["sentence"].astype(str).tolist()

aspect_probs = batched_aspect_probabilities(
    sentence_texts,
    batch_size=ASPECT_BATCH_SIZE,
)

print("Aspect probability matrix:", aspect_probs.shape)


## 6. Aspect Detection diagnosztika – itt álljunk meg először

In [ ]:
aspect_binary = (aspect_probs >= ASPECT_THRESHOLD).astype(int)
predicted_aspect_counts = aspect_binary.sum(axis=1)

aspect_detection_summary = pd.DataFrame({
    "metric": [
        "Összes mondat",
        "0 aspektus",
        "1 aspektus",
        "2 aspektus",
        "3+ aspektus",
        "Átlagos aspektusszám / mondat",
    ],
    "value": [
        len(sentences_df),
        int((predicted_aspect_counts == 0).sum()),
        int((predicted_aspect_counts == 1).sum()),
        int((predicted_aspect_counts == 2).sum()),
        int((predicted_aspect_counts >= 3).sum()),
        float(predicted_aspect_counts.mean()),
    ],
})

display(aspect_detection_summary)


In [ ]:
aspect_frequency = pd.DataFrame({
    "aspect": ASPECTS,
    "predicted_sentences": aspect_binary.sum(axis=0),
    "mean_probability": aspect_probs.mean(axis=0),
}).sort_values("predicted_sentences", ascending=False)

display(aspect_frequency)


### Első ellenőrzési pont

Itt érdemes először megállni. Azt ellenőrizzük, hogy az alkalmazási adatokon nem jelez-e a modell túl sok vagy túl kevés aspektust.


## 7. Mondat–aspektus párok létrehozása

In [ ]:
pair_rows = []

base_cols = [
    "place_index", "gmap_id", "name", "venue_type",
    "review_id", "review_rating", "sentence_id", "sentence"
]

for row_idx, row in sentences_df.iterrows():
    for aspect_idx, aspect in enumerate(ASPECTS):
        probability = aspect_probs[row_idx, aspect_idx]

        if probability < ASPECT_THRESHOLD:
            continue

        item = {col: row[col] for col in base_cols}
        item["aspect"] = aspect
        item["aspect_probability"] = float(probability)
        pair_rows.append(item)

sentence_aspect_df = pd.DataFrame(pair_rows)

print("Mondat–aspektus párok:", len(sentence_aspect_df))
print("Érintett mondatok:", sentence_aspect_df["sentence_id"].nunique())
display(sentence_aspect_df.head(15))


# 8. Sentiment Classification

In [ ]:
del aspect_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SENTIMENT_BATCH_SIZE = 64

sentiment_tokenizer = AutoTokenizer.from_pretrained(
    SENTIMENT_MODEL_DIR,
    local_files_only=True,
)

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL_DIR,
    local_files_only=True,
).to(DEVICE)

sentiment_model.eval()

sentiment_id2label = {
    int(k): v for k, v in sentiment_model.config.id2label.items()
}
SENTIMENT_LABELS = [
    sentiment_id2label[i]
    for i in range(len(sentiment_id2label))
]

print("Sentiment osztályok:", SENTIMENT_LABELS)


In [ ]:
sentiment_inputs = [
    f"Sentence: {sentence} Aspect: {aspect}."
    for sentence, aspect in zip(
        sentence_aspect_df["sentence"],
        sentence_aspect_df["aspect"],
    )
]

def batched_sentiment_probabilities(texts, batch_size=64):
    output = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = sentiment_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=160,
            return_tensors="pt",
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.inference_mode():
            logits = sentiment_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()

        output.append(probs)

    return np.vstack(output)

sentiment_probs = batched_sentiment_probabilities(
    sentiment_inputs,
    batch_size=SENTIMENT_BATCH_SIZE,
)

print("Sentiment probability matrix:", sentiment_probs.shape)


In [ ]:
sentiment_pred_ids = np.argmax(sentiment_probs, axis=1)

sentence_aspect_df["sentiment"] = [
    SENTIMENT_LABELS[i] for i in sentiment_pred_ids
]
sentence_aspect_df["sentiment_probability"] = sentiment_probs.max(axis=1)

for idx, label in enumerate(SENTIMENT_LABELS):
    sentence_aspect_df[f"prob_{label}"] = sentiment_probs[:, idx]

SENTIMENT_SCORE = {
    "negative": -1.0,
    "neutral": 0.0,
    "positive": 1.0,
}

sentence_aspect_df["sentiment_score"] = (
    sentence_aspect_df["sentiment"].map(SENTIMENT_SCORE)
)

display(sentence_aspect_df[[
    "name", "sentence", "aspect", "aspect_probability",
    "sentiment", "sentiment_probability"
]].head(20))


## 9. Sentiment diagnosztika

In [ ]:
sentiment_distribution = (
    sentence_aspect_df["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

sentiment_distribution["ratio"] = (
    sentiment_distribution["count"] / len(sentence_aspect_df)
)

display(sentiment_distribution)


# 10. Review-szintű aspektusprofil

Ugyanazon review-n belül az azonos aspektushoz tartozó több mondat sentimentjét előbb átlagoljuk.


In [ ]:
review_aspect_df = (
    sentence_aspect_df
    .groupby(
        [
            "place_index", "gmap_id", "name", "venue_type",
            "review_id", "aspect"
        ],
        as_index=False
    )
    .agg(
        review_aspect_sentiment=("sentiment_score", "mean"),
        sentence_mentions=("sentence_id", "nunique"),
        mean_aspect_probability=("aspect_probability", "mean"),
        mean_sentiment_probability=("sentiment_probability", "mean"),
    )
)

print("Review–aspect rekordok:", len(review_aspect_df))
display(review_aspect_df.head(15))


# 11. Helyszintű, 12 aspektusos profil

In [ ]:
place_aspect_profiles = (
    review_aspect_df
    .groupby(
        ["place_index", "gmap_id", "name", "venue_type", "aspect"],
        as_index=False
    )
    .agg(
        review_support=("review_id", "nunique"),
        sentence_mentions=("sentence_mentions", "sum"),
        mean_sentiment=("review_aspect_sentiment", "mean"),
        mean_aspect_probability=("mean_aspect_probability", "mean"),
        mean_sentiment_probability=("mean_sentiment_probability", "mean"),
    )
)

place_aspect_profiles["profile_score_0_100"] = (
    50 * (place_aspect_profiles["mean_sentiment"] + 1)
)

display(place_aspect_profiles.head(30))


# 12. Recommendation-dimenziók

A 12 SemEval aspektust magasabb szintű dimenziókba vonjuk össze:

- Food
- Drinks
- Service
- Ambience
- Price
- Location
- Overall
- Other


In [ ]:
ASPECT_TO_DIMENSION = {
    "FOOD#QUALITY": "Food",
    "FOOD#STYLE_OPTIONS": "Food",
    "DRINKS#QUALITY": "Drinks",
    "DRINKS#STYLE_OPTIONS": "Drinks",
    "SERVICE#GENERAL": "Service",
    "AMBIENCE#GENERAL": "Ambience",
    "FOOD#PRICES": "Price",
    "DRINKS#PRICES": "Price",
    "RESTAURANT#PRICES": "Price",
    "LOCATION#GENERAL": "Location",
    "RESTAURANT#GENERAL": "Overall",
    "RESTAURANT#MISCELLANEOUS": "Other",
}

review_aspect_df["dimension"] = (
    review_aspect_df["aspect"].map(ASPECT_TO_DIMENSION)
)

assert review_aspect_df["dimension"].notna().all()


## 12.1. Review × dimenzió aggregálás

In [ ]:
review_dimension_df = (
    review_aspect_df
    .groupby(
        [
            "place_index", "gmap_id", "name", "venue_type",
            "review_id", "dimension"
        ],
        as_index=False
    )
    .agg(
        review_dimension_sentiment=("review_aspect_sentiment", "mean"),
        aspect_mentions=("aspect", "nunique"),
    )
)

display(review_dimension_df.head(15))


# 13. Helyszintű recommendation profile

In [ ]:
place_dimension_profiles = (
    review_dimension_df
    .groupby(
        ["place_index", "gmap_id", "name", "venue_type", "dimension"],
        as_index=False
    )
    .agg(
        review_support=("review_id", "nunique"),
        mean_sentiment=("review_dimension_sentiment", "mean"),
    )
)

place_dimension_profiles["profile_score_0_100"] = (
    50 * (place_dimension_profiles["mean_sentiment"] + 1)
)

display(place_dimension_profiles.head(30))


## 13.1. Széles place profile tábla

In [ ]:
index_cols = ["place_index", "gmap_id", "name", "venue_type"]

score_wide = (
    place_dimension_profiles
    .pivot(
        index=index_cols,
        columns="dimension",
        values="profile_score_0_100",
    )
    .add_suffix("_score")
    .reset_index()
)

support_wide = (
    place_dimension_profiles
    .pivot(
        index=index_cols,
        columns="dimension",
        values="review_support",
    )
    .add_suffix("_support")
    .reset_index()
)

place_profiles_wide = score_wide.merge(
    support_wide,
    on=index_cols,
    how="outer",
)

metadata_cols = [
    col for col in [
        "gmap_id", "address", "category_text",
        "avg_rating", "num_of_reviews", "price"
    ]
    if col in places_df.columns
]

place_profiles_wide = (
    place_profiles_wide
    .merge(
        places_df[metadata_cols],
        on="gmap_id",
        how="left",
    )
    .sort_values("place_index")
)

display(place_profiles_wide)


## 14. Profilok és coverage ellenőrzése

In [ ]:
MAIN_DIMENSIONS = [
    "Food", "Drinks", "Service",
    "Ambience", "Price", "Location", "Overall"
]

display_cols = ["place_index", "name", "venue_type"]

for dimension in MAIN_DIMENSIONS:
    for suffix in ["score", "support"]:
        col = f"{dimension}_{suffix}"
        if col in place_profiles_wide.columns:
            display_cols.append(col)

display(place_profiles_wide[display_cols].round(1))


In [ ]:
coverage_rows = []

for dimension in MAIN_DIMENSIONS:
    score_col = f"{dimension}_score"
    support_col = f"{dimension}_support"

    if score_col not in place_profiles_wide.columns:
        continue

    coverage_rows.append({
        "dimension": dimension,
        "places_with_score": int(place_profiles_wide[score_col].notna().sum()),
        "median_review_support": float(place_profiles_wide[support_col].median()),
        "min_review_support": float(place_profiles_wide[support_col].min()),
        "max_review_support": float(place_profiles_wide[support_col].max()),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)


## 14.1. Support-adjusted place profile-ok

A kevés review-ból származó szélsőséges score-ok stabilizálására egyszerű shrinkage-et alkalmazunk. Minden nyers dimenzió-score-t 50-es semleges prior felé húzunk, a prior erőssége 10 review.

`adjusted = (support × raw_score + 10 × 50) / (support + 10)`

Így a nagy supporttal rendelkező dimenziók alig változnak, a nagyon ritka dimenziók viszont nem kapnak indokolatlanul szélsőséges értéket.

In [ ]:
PRIOR_MEAN = 50.0
PRIOR_STRENGTH = 10.0

for dimension in MAIN_DIMENSIONS:
    score_col = f"{dimension}_score"
    support_col = f"{dimension}_support"
    adjusted_col = f"{dimension}_adjusted"

    if score_col not in place_profiles_wide.columns:
        continue

    score = place_profiles_wide[score_col].fillna(PRIOR_MEAN)
    support = place_profiles_wide[support_col].fillna(0.0)

    place_profiles_wide[adjusted_col] = (
        support * score
        + PRIOR_STRENGTH * PRIOR_MEAN
    ) / (
        support + PRIOR_STRENGTH
    )

adjusted_cols = [
    f"{dimension}_adjusted"
    for dimension in MAIN_DIMENSIONS
    if f"{dimension}_adjusted" in place_profiles_wide.columns
]

display(
    place_profiles_wide[
        ["place_index", "name", "venue_type"] + adjusted_cols
    ].round(1)
)

# 15. Export

In [ ]:
PREDICTIONS_PATH = Path("sentence_aspect_sentiment_predictions.csv")
REVIEW_ASPECT_PATH = Path("review_aspect_profiles.csv")
PLACE_ASPECT_PATH = Path("place_aspect_profiles.csv")
PLACE_PROFILE_PATH = Path("place_profiles_wide.csv")

sentence_aspect_df.to_csv(PREDICTIONS_PATH, index=False)
review_aspect_df.to_csv(REVIEW_ASPECT_PATH, index=False)
place_aspect_profiles.to_csv(PLACE_ASPECT_PATH, index=False)
place_profiles_wide.to_csv(PLACE_PROFILE_PATH, index=False)

print("Elmentve:")
print("-", PREDICTIONS_PATH)
print("-", REVIEW_ASPECT_PATH)
print("-", PLACE_ASPECT_PATH)
print("-", PLACE_PROFILE_PATH)


## 16. Fájlok letöltése

In [ ]:
try:
    from google.colab import files
    files.download(str(PREDICTIONS_PATH))
    files.download(str(REVIEW_ASPECT_PATH))
    files.download(str(PLACE_ASPECT_PATH))
    files.download(str(PLACE_PROFILE_PATH))
except ImportError:
    print("A CSV-k a notebook mappájában találhatók.")


# Következő lépés

Ha ez a notebook rendben lefutott, akkor a `place_profiles_wide.csv` közvetlenül használható a **06 – Personalized Recommendation** részhez.

Ott a természetes nyelvű felhasználói preferenciákat a place profile dimenzióival vetjük össze, és Top-N ajánlást készítünk.
